In [ ]:
import ast
import sys
from datetime import datetime, timezone
from pathlib import Path

import inflect
import pandas as pd
import requests
from kaleido.scopes.plotly import PlotlyScope
from tqdm import tqdm

sys.path.append(str(Path("..").resolve()))

# Surpress NLP Mask Warning for Apple Silicon
import warnings

import indicator
import literature
import novel
import score

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r".*encoder_attention_mask.*BertSdpaSelfAttention\.forward"
)

## Curate Literature (DGIdb Control)

In [ ]:
publications = pd.read_csv('data/dgidb/publications.csv')
pmids = publications['pmid'].tolist()
pmids = [str(pmid) for pmid in pmids]
abstracts = literature.fetch_abstracts(pmids)
abstracts[0:5]

In [ ]:
abstract = pd.DataFrame(abstracts, columns=["Title", "Abstract"])
abstract

In [ ]:
results = novel.batch(abstract['Abstract'])
results

In [ ]:
results[results['entity_group'] == 'CHEMICAL']

In [ ]:
def _singularize(word: str) -> str:
    """Convert a plural word into a singular word."""
    inflector = inflect.engine()
    return inflector.singular_noun(word) or word

def _normalize_therapy(word: str) -> list[str, str, str]:
    try:
        r = requests.get(
            f"https://normalize.cancervariants.org/therapy/normalize?q={word}&infer_namespace=true",
            timeout=10
        )
        r.raise_for_status()
        response = r.json()

        if isinstance(response, dict) and response.get("match_type") is not None:
            if response["match_type"] != 0:
                return [
                    response["match_type"],
                    response["therapy"]["id"],
                    response["therapy"]["name"],
                ]
            return [0, None, None]  # Not matched

        return ["Unexpected Response Format", None, None] # noqa: TRY300

    except requests.exceptions.RequestException as e:
        return ["HTTP Error", str(e), None]
    except Exception as e:
        return ["Failure to Normalize", str(e), None]

# Main loop
checkpoint_interval = 5000
output_base = "normalized_results_checkpoint"
for idx, (index, row) in enumerate(tqdm(results[results['entity_group'] == 'CHEMICAL'].iterrows()), 1):
    word = _singularize(row['word'])
    norm_result = _normalize_therapy(word)

    results.at[index, 'concept_match_type'] = norm_result[0]
    results.at[index, 'concept_id'] = norm_result[1]
    results.at[index, 'concept_label'] = norm_result[2]

    if idx % checkpoint_interval == 0:
        checkpoint_filename = f"{output_base}_checkpoint_{idx}.xlsx"
        results.to_excel(checkpoint_filename, index=False)
        print(f"Checkpoint saved at row {idx} -> {checkpoint_filename}") # noqa: T201

# Final save after loop completes
final_filename = f"{output_base}_final.xlsx"
# results.to_excel(final_filename, index=False)

In [ ]:
results = pd.read_excel('normalized_results_checkpoint_final.xslx')

In [ ]:
results = pd.read_excel('control_test3_normalized_results_final.xlsx')
tdf = results[(results['concept_match_type']!=0) & (results['concept_match_type'].isna()==False)].reset_index(drop=True) # noqa: E712
condensed_results = tdf.groupby('original_text').apply(
    lambda group: pd.Series({
        'DRUG_LABELS': ' | '.join(group.loc[group['entity_group'] == 'CHEMICAL', 'concept_label'].dropna().astype(str).unique()),
        'DRUG_IDS': ' | '.join(group.loc[group['entity_group'] == 'CHEMICAL', 'concept_id'].dropna().astype(str).unique())
    })
).reset_index()

condensed_results

In [ ]:
merged_df = pd.merge(
    abstract,
    condensed_results,
    left_on='Abstract',
    right_on='original_text',
    how='left'
)
merged_df = merged_df[merged_df['DRUG_LABELS'].isnull()==False].reset_index(drop=True) # noqa: E712
merged_df

### Generate Score

In [ ]:
dgidb_df = pd.read_csv('search/2025-08-13_BCL2_clin_score.csv')
dgidb_df = dgidb_df.drop_duplicates(subset=['Drug','Gene'], keep='first')
dgidb_df.head()

In [ ]:
import importlib

importlib.reload(indicator)

indicator.generate_interaction_evidence(merged_df, dgidb_df)

## Load Scores

In [ ]:
tdf = score.load_pmid_assessments('2025-08-25_BCL2.zip', 'interaction_search')
tdf = tdf[tdf['label']=='interaction_evidence'].reset_index(drop=True)
tdf


In [ ]:
def unpack_total(score: float|None) -> int:
    """Extract the 'unweighted total' score from a serialized score object."""
    if type(score) is float:
        return 0
    if score is None:
        return 0
    return ast.literal_eval(score)['unweighted_total']

def unpack_regulation(score: float|None) -> int:
    """Extract the 'regulation_changes' score from a serialized score object."""
    if type(score) is float:
        return 0
    if score is None:
        return 0
    return ast.literal_eval(score)['regulation_changes']

tdf['total_interaction_evidence'] = tdf['scores'].apply(unpack_total)
tdf['total_regulation_evidence'] = tdf['scores'].apply(unpack_total)


tdf.sort_values(by='total_regulation_evidence', ascending=False)[0:10]

### Build Prompts

In [ ]:
here_we_go = tdf.sort_values(by='total_interaction_evidence', ascending=False)[0:100].reset_index(drop=True)

def generate_prompt_base(drugs: str) -> str:
    """Generate a standardized prompt for extracting drug-gene interactions."""
    prompt = f"""You are an expert biomedical scientist, biochemist, and scientific curator trained to identify drug-gene interactions from scientific literature. Given a list of drugs, a gene, and a scientific abstract, your task is to determine whether an interaction between a drug and a gene is occuring and assign it an interaction directionality. Use the following tools to help perform these tasks.

    *Interaction*
    - An interaction between a small molecule and a gene or gene product.

    *Interaction Directionality*
    - Activating -> Activating interactions are those where the drug increases the biological activity or expression of a gene target.
    - Inhibiting -> Inhibiting interactions are those where the drug decreases the biological activity or expression of a gene target.

    The drugs to consider for this task are {drugs}. For each drug, fill out the following JSON schema:
    {{
        "pmid": "EXACT PMID FROM CONTEXT",
        "drug_name": "NAME OF DRUG",
        "gene_name": "NAME OF GENE BEING INTERACTED WITH",
        "interaction_occurs_with_gene": "YES" or "NO",
        "interaction_type": "ACTIVATING" or "INHIBITING",
        "evidence": "EXACT SENTENCE FROM ABSTRACT THAT SUPPORTS INTERACTION"
    }}
"""
    return prompt # noqa: RET504 (readability)

here_we_go['context'] = None
here_we_go['prompt'] = None

for idx, row in here_we_go.iterrows():
    drugs = row['tagged_drugs']
    gene = row['gene']
    prompt_base = generate_prompt_base(drugs, gene)

    # tdf = here_we_go[here_we['pmid']==row['pmid']].reset_index(drop=True)
    context = f'PMID: {row['pmid']}\n Abstract: {row['abstract']}'

    full_prompt = f'{prompt_base}\n\n{context}'

    here_we_go.at[idx,'context'] = context
    here_we_go.at[idx,'prompt'] = full_prompt



here_we_go.head()

In [ ]:
import boto3

# Initialize the Bedrock Runtime client
bedrock = boto3.client("bedrock-runtime", region_name="us-east-2")

# Replace with your actual inference profile ID or ARN
INFERENCE_PROFILE_ID = "us.anthropic.claude-3-5-sonnet-20240620-v1:0"
# Not opus 4 I guess????


def query_claude_sonnet(prompt: str) -> str:
    """Send a prompt to the Claude Sonnet model via AWS Bedrock and return the response text."""
    body = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1024,
        "temperature": 0.0,
        "anthropic_version": "bedrock-2023-05-31"
    }

    try:
        response = bedrock.invoke_model(
            body=json.dumps(body),
            modelId=INFERENCE_PROFILE_ID,
            contentType="application/json",
            accept="application/json"
        )
        response_body = json.loads(response["body"].read())
        return response_body["content"][0]["text"]
    except Exception as e:
        return f"[Error] {e!s}"

# Example usage
# response = query_claude_opus("Hello! This is a test message!")
# print(response)


In [ ]:
here_we_go['response'] = None

for idx, row in here_we_go.iterrows():
    here_we_go.at[idx,'response'] = query_claude_sonnet(row['prompt'])

In [ ]:
# here_we_go.to_excel('Literature_Prioritization_control3_use_this.xlsx')

### Extract!

In [ ]:
import json
import logging
import re

logger = logging.getLogger(__name__)

def extract_json_objects(text: str) -> list[dict]:
    """Extract all JSON-like objects (dicts) from a text string."""
    objects: list[dict] = []
    brace_stack: list[str] = []
    start: int | None = None

    for i, ch in enumerate(text):
        if ch == "{":
            if not brace_stack:
                start = i
            brace_stack.append("{")

        elif ch == "}":
            if brace_stack:
                brace_stack.pop()

                if not brace_stack and start is not None:
                    snippet = text[start:i + 1]

                    try:
                        objects.append(json.loads(snippet))

                    except json.JSONDecodeError as e:
                        # fallback: strip trailing commas etc.
                        cleaned = re.sub(r",\s*}", "}", snippet)
                        cleaned = re.sub(r",\s*]", "]", cleaned)

                        try:
                            objects.append(json.loads(cleaned))

                        except json.JSONDecodeError as e2:
                            logger.debug(
                                "Failed to parse JSON snippet. "
                                f"Original error: {e}; After cleanup: {e2}"
                            )

                    start = None

    return objects


def extract_note_text(text: str) -> str:
    """Grab any trailing 'Note:' or 'Explanation:' text from a response.
    Returns a single string ('' if none found).
    """
    match = re.search(r"(?:Note|Explanation)[:\-–]\s*(.+)", text, flags=re.IGNORECASE|re.DOTALL) # noqa: RUF001
    return match.group(1).strip() if match else ""


In [ ]:
here_we_go['json'] = None
here_we_go['free_text_explanation'] = None
for idx, row in here_we_go.iterrows():
    here_we_go.at[idx, 'free_text_explanation'] = extract_note_text(row['response'])
    here_we_go.at[idx,'json'] = extract_json_objects(row['response'])

here_we_go.head()

In [ ]:
here_we_go['json'][0]

In [ ]:
flat_df = pd.DataFrame([obj for lst in here_we_go["json"] for obj in lst])
flat_df


In [ ]:
flat_df = flat_df[(flat_df['interaction_occurs_with_gene']=='YES') & (flat_df['gene_name']!='N/A')]
flat_df

In [ ]:
def _normalize_gene(word: str) -> list[str, str, str]:
    try:
        r = requests.get(
            f'https://normalize.cancervariants.org/gene/normalize?q={word}',
            timeout=10  # Set timeout for network reliability
        )
        r.raise_for_status()
        response = r.json()

        if isinstance(response, dict) and response.get('match_type') is not None:
            if response['match_type'] != 0:
                return [
                    response['match_type'],
                    response['gene']['id'],
                    response['gene']['name']
                ]
            return [0, None, None]  # Not matched
        return ['Unexpected Response Format', None, None] # noqa: TRY300
    except requests.exceptions.RequestException as e:
        return ['HTTP Error', str(e), None]
    except Exception as e:
        return ['Failure to Normalize', str(e), None]

def _normalize_therapy(word: str) -> list[str, str, str]:
    try:
        r = requests.get(
            f"https://normalize.cancervariants.org/therapy/normalize?q={word}&infer_namespace=true",
            timeout=10
        )
        r.raise_for_status()
        response = r.json()

        if isinstance(response, dict) and response.get("match_type") is not None:
            if response["match_type"] != 0:
                return [
                    response["match_type"],
                    response["therapy"]["id"],
                    response["therapy"]["name"],
                ]
            return [0, None, None]  # Not matched

        return ["Unexpected Response Format", None, None] # noqa: TRY300

    except requests.exceptions.RequestException as e:
        return ["HTTP Error", str(e), None]
    except Exception as e:
        return ["Failure to Normalize", str(e), None]


In [ ]:
flat_df['gene_concept'] = None
flat_df['gene_label'] = None
flat_df['gene_match_type'] = None
flat_df['drug_concept'] = None
flat_df['drug_label'] = None
flat_df['drug_match_type'] = None
for idx, row in tqdm(flat_df.iterrows()):
    drug_match_type, drug_concept, drug_label = _normalize_therapy(row['drug_name'])
    gene_match_type, gene_concept, gene_label = _normalize_gene(row['gene_name'])

    flat_df.at[idx, 'gene_concept'] = gene_concept
    flat_df.at[idx, 'gene_label'] = gene_label
    flat_df.at[idx, 'gene_match_type'] = gene_match_type
    flat_df.at[idx, 'drug_concept'] = drug_concept
    flat_df.at[idx, 'drug_label'] = drug_label
    flat_df.at[idx, 'drug_match_type'] = drug_match_type

flat_df

In [ ]:
flat_df['gene_concept'].value_counts(dropna=False)

In [ ]:
flat_df['drug_concept'].value_counts(dropna=False)

In [ ]:
final_results = flat_df[(flat_df['gene_concept'].isna()==False)].reset_index(drop=True) # noqa: E712
final_results

In [ ]:
188 / 212

In [ ]:
# final_results.to_excel('final_results_control_test3.xlsx')

### Graph

In [ ]:
import pandas as pd

final_results = pd.read_excel('literature_curation/final_results_control_test3.xlsx')

In [ ]:
import pandas as pd
import psycopg2

# 1) Prepare PMIDs from your DataFrame
pmids = pd.Series(final_results['pmid']).dropna().astype(str).unique().tolist()

# 2) Connect
conn = psycopg2.connect(
    dbname="dgidb_2025",
    user="mjc014",
    password="",
    host="localhost",
    port="5432"
)

# 3) Query: cast column -> text so it matches the text[] param
query = """
SELECT *
FROM public.publications
WHERE pmid::text = ANY(%s)
ORDER BY id ASC;
"""

df = pd.read_sql(query, conn, params=(pmids,))
conn.close()

print(f"Found {len(df)} PMIDs already in DGIdb") # noqa: T201
display(df.head())

# 4) Novelty %
found_pmids = set(df['pmid'].astype(str).unique())
all_pmids   = set(pmids)
novel_pmids = sorted(all_pmids - found_pmids)

novelty_pct = 100.0 * (len(novel_pmids) / max(1, len(all_pmids)))
print(f"Novel PMIDs: {len(novel_pmids)} / {len(all_pmids)} = {novelty_pct:.2f}%") # noqa: T201


In [ ]:

import plotly.express as px

# Data summary
counts = {
    "Novel PMIDs": len(novel_pmids),
    "Known PMIDs (in DGIdb)": len(found_pmids)
}

# Create dataframe
plot_df = pd.DataFrame(list(counts.items()), columns=["Category", "Count"])

# Build pie chart
fig = px.pie(
    plot_df,
    names="Category",
    values="Count",
    color="Category",
    color_discrete_map={
        "Novel PMIDs": "#1f77b4",            # blue
        "Known PMIDs (in DGIdb)": "#ff7f0e"  # orange
    },
    hole=0.35  # makes it a donut chart, easier to read in talks/posters
)

# Update layout for professional styling
fig.update_traces(
    textinfo="label+percent",   # show both category and %
    textfont_size=16,
    pull=[0.05, 0]  # gently pull out the "Novel" slice for emphasis
)

fig.update_layout(
    title={
        "text":"Novelty of PMIDs Compared to DGIdb",
        "font":{"size":22, "family":"Arial, sans-serif"},
        "x":0.5, "xanchor":"center"
    },
    legend={
        "font":{"size":14},
        "orientation":"h",
        "yanchor":"bottom",
        "y":-0.15,
        "xanchor":"center",
        "x":0.5
    },
    margin={"t":60, "b":60, "l":40, "r":40}
)

fig.show()

# Save as high-res PNG
fig.write_image("pmid_control_novelty_pie.png",
                engine="kaleido",
                width=1600, height=1200, scale=3)

# Vector versions (great for posters)
fig.write_image("pmid_control_novelty_pie.svg", engine="kaleido")
fig.write_image("pmid_control_novelty_pie.pdf", engine="kaleido")



In [ ]:
from collections import defaultdict
from collections.abc import Iterator, Sequence

import pandas as pd

# -------- Inputs --------
df_in = final_results.copy()
df_in["gene_label"] = df_in["gene_label"].astype(str).str.strip()
df_in["drug_label"] = df_in["drug_label"].astype(str).str.strip()

# Unique pairs and gene list
pairs = df_in[["gene_label", "drug_label"]].dropna().drop_duplicates()
genes = sorted(pairs["gene_label"].unique().tolist())

GQL_URL = "https://dgidb.org/api/graphql"

GQL_QUERY = """
query GeneInteractions($genes: [String!]!) {
  genes(names: $genes) {
    nodes {
      name
      conceptId
      interactions {
        drug {
          name
          conceptId
          approved
        }
        gene {
          name
          conceptId
          longName
        }
        interactionScore
        interactionTypes { type directionality }
        interactionAttributes { name value }
        publications { pmid }
        sources { sourceDbName }
      }
    }
  }
}
"""

def batched(seq: Sequence, n:int=40) -> Iterator[Sequence]:
    """Yield successive chunks from a sequence."""
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

# gene -> set of lowercase drug names that DGIdb reports for that gene
gene_to_known_drugs = defaultdict(set)
session = requests.Session()
session.headers.update({"Accept": "application/json"})

for chunk in batched(genes, 40):
    resp = session.post(GQL_URL, json={"query": GQL_QUERY, "variables": {"genes": chunk}}, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if "errors" in data:
        raise RuntimeError(data["errors"])
    nodes = (data.get("data", {}).get("genes", {}) or {}).get("nodes", []) or []
    for node in nodes:
        gname = (node.get("name") or "").strip()
        if not gname:
            continue
        known = gene_to_known_drugs[gname]
        for itx in node.get("interactions", []) or []:
            drug_obj = itx.get("drug") or {}
            # Prefer structured drug.name; fall back if older fields exist
            dname = (drug_obj.get("name")
                     or itx.get("drugName")  # backward compat (older examples)
                     or "").strip()
            if dname:
                known.add(dname.lower())

# Flag known vs novel (case-insensitive drug name check per gene)
pairs["known_in_dgidb"] = pairs.apply(
    lambda r: r["drug_label"].lower() in gene_to_known_drugs.get(r["gene_label"], set()),
    axis=1
)

# Metrics
n_total = len(pairs)
n_known = int(pairs["known_in_dgidb"].sum())
n_novel = int((~pairs["known_in_dgidb"]).sum())
pct_novel = (100.0 * n_novel / n_total) if n_total else 0.0

print(f"Unique gene-drug pairs: {n_total}") # noqa: T201
print(f"Known in DGIdb:         {n_known}") # noqa: T201
print(f"Novel (not in DGIdb):   {n_novel}") # noqa: T201
print(f"Novelty = {pct_novel:.2f}%") # noqa: T201

# Inspect a few novel pairs
pairs.loc[~pairs["known_in_dgidb"]].sort_values(["gene_label","drug_label"]).head(10)


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# ---- Inputs: counts from your earlier classification ----
n_total = len(pairs)
n_known = int(pairs["known_in_dgidb"].sum())
n_novel = int((~pairs["known_in_dgidb"]).sum())

pct_known = (100.0 * n_known / n_total) if n_total else 0.0
pct_novel = (100.0 * n_novel / n_total) if n_total else 0.0

# ---- Data frame (for clarity; not strictly required) ----
df_plot = pd.DataFrame({
    "Category": ["Novel Interactions", "Known Interactions (DGIdb)"],
    "Count": [n_novel, n_known],
    "Percent": [pct_novel, pct_known],
})

# ---- Colors (high-contrast, colorblind-friendly) ----
COLOR_NOVEL = "#1f77b4"   # blue
COLOR_KNOWN = "#ff7f0e"   # orange

# ---- Build a 100% stacked horizontal bar ----
fig = go.Figure()

fig.add_trace(go.Bar(
    x=[pct_novel], y=["Interactions"],
    orientation="h",
    name="Novel",
    marker_color=COLOR_NOVEL,
    text=[f"{pct_novel:.1f}% ({n_novel:,})"],
    textposition="inside",
    textfont={"size":16},
    hovertemplate="Novel: %{x:.1f}%<br>Count: " + f"{n_novel:,}" + "<extra></extra>"
))

fig.add_trace(go.Bar(
    x=[pct_known], y=["Interactions"],
    orientation="h",
    name="Known in DGIdb",
    marker_color=COLOR_KNOWN,
    text=[f"{pct_known:.1f}% ({n_known:,})"],
    textposition="inside",
    textfont={"size":16},
    hovertemplate="Known: %{x:.1f}%<br>Count: " + f"{n_known:,}" + "<extra></extra>"
))

fig.update_layout(
    barmode="stack",
    xaxis={"range":[0, 100], "title":None, "showgrid":False, "showticklabels":False},
    yaxis={"title":None, "showgrid":False},
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend={
        "orientation":"h",
        "yanchor":"bottom", "y":-0.15,
        "xanchor":"center", "x":0.5,
        "font":{"size":14}
    },
    margin={"t":80, "b":80, "l":40, "r":40},
    title={
        "text": "Novelty of Gene-Drug Interactions vs DGIdb",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 22, "family": "Arial, sans-serif"}
    },
    font={"family": "Arial, sans-serif", "size": 16},  # ← fixed
)

# Subtitle-like annotation with totals and date
fig.add_annotation(
    x=0.5, y=-0.35, xref="paper", yref="paper", showarrow=False,
    text=f"Total unique pairs: {n_total:,} • Generated {datetime.now(timezone.utc).date().isoformat()}",
    font={"size":14, "color":"gray"}
)

fig.show()

# ---- High-quality export (PNG/SVG/PDF) via PlotlyScope ----

scope = PlotlyScope()

png = scope.transform(fig.to_dict(), format="png", width=1600, height=900, scale=3)
with Path("interaction_control_novelty_100pct_bar.png").open("wb") as f:
    f.write(png)

svg = scope.transform(fig.to_dict(), format="svg", width=1600, height=900, scale=3)
with Path("interaction_control_novelty_100pct_bar.svg").open("wb") as f:
    f.write(svg)

pdf = scope.transform(fig.to_dict(), format="pdf", width=1600, height=900, scale=3)
with Path("interaction_control_novelty_100pct_bar.pdf").open("wb") as f:
    f.write(pdf)
